In [31]:
import pandas as pd
import unicodedata
import re

In [32]:
df_saber11 = pd.read_csv(r"C:\Proyectos Personales\colombia icfes analytics\data\data_raw\resultados_icfes\Examen_Saber_11_20252.txt", sep=";")
df_dim_estudiante = pd.read_csv(r"C:\Proyectos Personales\colombia icfes analytics\data\data_cleaned\dim_estudiante.csv")
df_dim_colegio = pd.read_csv(r"C:\Proyectos Personales\colombia icfes analytics\data\data_cleaned\dim_colegio.csv")
df_dim_divipola = pd.read_csv(r"C:\Proyectos Personales\colombia icfes analytics\data\data_cleaned\dim_divipola.csv")

C:\Users\estebanab\AppData\Local\Temp\ipykernel_21132\4220401968.py:1: DtypeWarning: Columns (0: cole_area_ubicacion, 1: cole_bilingue, 2: cole_calendario, 3: cole_caracter, 4: cole_depto_ubicacion, 5: cole_genero, 6: cole_jornada, 7: cole_mcpio_ubicacion, 8: cole_naturaleza, 9: cole_nombre_establecimiento, 10: cole_nombre_sede, 11: cole_sede_principal, 12: fami_numhermanos, 13: estu_comunidadcampesina, 14: estu_numhijos, 15: estu_horastrabnoremu, 16: fami_posicionhermanos, 17: estu_tiempocasaacole, 18: estu_desplazacolegio) have mixed types. Specify dtype option on import or set low_memory=False.
  df_saber11 = pd.read_csv(r"C:\Proyectos Personales\colombia icfes analytics\data\data_raw\resultados_icfes\Examen_Saber_11_20252.txt", sep=";")


In [33]:
def limpiar_columna(nombre):
    nombre = unicodedata.normalize('NFKD', nombre).encode('ascii', 'ignore').decode('utf-8')
    nombre = nombre.strip()
    nombre = re.sub(r'\s+', '_', nombre)
    nombre = nombre.replace('"', '').replace("'", '')
    return nombre.upper()

df_saber11.columns = [limpiar_columna(col) for col in df_saber11.columns]

In [34]:
#============================== Creación de la tabla fact_icfes ==============================

df_fact = df_saber11[['PERIODO','COLE_COD_DEPTO_UBICACION', 'COLE_COD_MCPIO_UBICACION','ESTU_CONSECUTIVO','COLE_CODIGO_ICFES','PUNT_C_NATURALES', 'PUNT_LECTURA_CRITICA', 'PUNT_MATEMATICAS', 'PUNT_SOCIALES_CIUDADANAS', 'PUNT_INGLES', 'PUNT_GLOBAL']]

df_fact["COLE_CODIGO_ICFES"] = df_fact["COLE_CODIGO_ICFES"].astype('Int64')

#======== unión de claves foráneas con las dimensiones =================

df_fact = df_fact.merge(df_dim_estudiante[['ESTU_ID', 'ESTU_SK']], left_on='ESTU_CONSECUTIVO', right_on='ESTU_ID', how='left') \
                 .merge(df_dim_colegio[['COLE_ID', 'COLE_SK']], left_on='COLE_CODIGO_ICFES', right_on='COLE_ID', how='left') \
                 .merge(df_dim_divipola[['CODIGO_MUNICIPIO', 'DIVIPOLA_SK']], left_on='COLE_COD_MCPIO_UBICACION', right_on='CODIGO_MUNICIPIO', how='left')

#======== Eliminar columnas inncesarias de la tabla de hechos =================

df_fact = df_fact.drop(['ESTU_CONSECUTIVO', 'ESTU_ID', 'COLE_CODIGO_ICFES', 'COLE_ID', 'COLE_COD_DEPTO_UBICACION', 'CODIGO_MUNICIPIO','COLE_COD_MCPIO_UBICACION'], axis=1)

df_fact.to_csv(r"C:\Proyectos Personales\colombia icfes analytics\data\data_cleaned\fact_icfes.csv", index=False)